## Section 0 — Setup

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U transformers datasets peft wandb scikit-learn torchao

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

# ---- Project config -------------------------------------------------------
WANDB_PROJECT = "24f2003563-t22026"
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OUT_DIR = "/kaggle/working"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

print("Train:", train.shape, "| Test:", test.shape)
train.head()


In [ ]:
# Quick sanity checks worth putting in the report's "Dataset" section.
print("Missing values in train:\n", train.isnull().sum().sum())
print("Answer options seen:", sorted(train['answer'].unique()))
print("Every question has exactly 5 options (A-E), 1 of which is correct ->",
      "expected positive rate per row ≈ 0.20")

## Exploratory Data Analysis

A quick look at the data before modeling (Report Section 3): how long the text is, and whether classes are balanced.

In [ ]:
# make a row on each option with question like question-option


def build_dataset(df, has_answer = True):
    """Explode each question into 5 rows (one per option A-E)."""
    rows = []
    for _, row in df.iterrows():
        for option in ['A', 'B', 'C', 'D', 'E']:
            item = {
                "id": row['id'],
                "prompt": row['prompt'],
                "option_text": row[option],
                "text": f"Question: {row['prompt']} Candidate Answer: {row[option]}",
                "option": option,
            }
            if has_answer:
                item["label"] = 1 if option == row['answer'] else 0
            rows.append(item)
    return pd.DataFrame(rows)

train_processed = build_dataset(train, has_answer=True)
test_processed  = build_dataset(test,  has_answer=False)
print(train_processed.shape, test_processed.shape)

In [ ]:
train_processed["word_len"] = train_processed["text"].str.split().apply(len)

print("Average words per row:", round(train_processed["word_len"].mean(), 1))
print("Longest row (words):", train_processed["word_len"].max())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(train_processed["word_len"], bins=20, color="#4C72B0")
axes[0].set_title("How long is each row?")
axes[0].set_xlabel("Words")

counts = train_processed["label"].value_counts().sort_index()
axes[1].bar(["Wrong (0)", "Correct (1)"], counts.values, color=["#C44E52", "#55A868"])
axes[1].set_title("Class balance (4 wrong : 1 correct)")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_summary.png", dpi=150)
plt.show()


In [ ]:
# Split by QUESTION id (not row!) so all 5 options of a question stay together
train_ids, val_ids = train_test_split(train['id'].unique(), test_size=0.15, random_state=42)

train_df = train_processed[train_processed['id'].isin(train_ids)].reset_index(drop=True)
val_df   = train_processed[train_processed['id'].isin(val_ids)].reset_index(drop=True)
val_answers = train[train['id'].isin(val_ids)].set_index("id")["answer"]

print(f"Train questions: {len(train_ids)} | Val questions: {len(val_ids)}")


In [ ]:
# map@3 is Kaggle leaderboard metrics for this competition
# scores_to_submission to select top three predictions and return it according to map@3 format
# normalization for range values of different models to 0 to 1
# evaluate model to give score to prediction map@3 and accuracy/F1 for each predictions

def map_at_3(true_answers_by_id, submission_df):
    # map@3 for score for each prediction and then taking average of all predictions
    scores = []
    for _, row in submission_df.iterrows():
        true = true_answers_by_id.loc[row["ID"]]
        preds = row["Prediction"].split()[:3]
        score = 0.0
        for i, p in enumerate(preds):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores))


def scores_to_submission(id_list, option_list, score_list, out_path = None):
    # Turn per-option scores into a ranked top-3 submission (ID, Prediction)
    df = pd.DataFrame({"id": id_list, "option": option_list, "score": score_list})
    rows = [
        {"ID": qid, "Prediction": " ".join(g.sort_values("score", ascending=False)["option"].values[:3])}
        for qid, g in df.groupby("id")
    ]
    sub = pd.DataFrame(rows)
    if out_path:
        sub.to_csv(out_path, index=False)
    return sub


def normalize(scores):
    # Min-max scale so scores from very different models can be averaged fairly
    scores = np.array(scores, dtype=float)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)


def evaluate_model(name, df, scores, true_answers_by_id):
    """
    MAP@3       : official competition metric (top-3 ranking)
    Accuracy/F1 : treat the highest-scored option as the single prediction, and compare it to the true answer.
    """
    work = df.reset_index(drop=True).copy()
    work["score"] = np.asarray(scores)

    sub = scores_to_submission(work["id"].values, work["option"].values, work["score"].values)
    map3 = map_at_3(true_answers_by_id, sub)

    work["pred_label"] = 0
    top_idx = work.groupby("id")["score"].idxmax()
    work.loc[top_idx, "pred_label"] = 1

    acc = accuracy_score(work["label"], work["pred_label"])
    f1 = f1_score(work["label"], work["pred_label"], zero_division=0)

    metrics = {"val_map3": map3, "val_accuracy": acc, "val_f1": f1}
    print(f"{name:32s} | MAP@3={map3:.4f} | Accuracy={acc:.4f} | F1={f1:.4f}")
    return metrics


## W&B Login

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

def log_run(run_name, metrics, config = None):
    """Log one finished model's metrics as its own W&B run"""
    run = wandb.init(project=WANDB_PROJECT, name=run_name, config=config or {})
    wandb.log(metrics)
    run.finish()


## Model 1: Fine-Tuned Transformer - DeBERTa-v3 + LoRA (Report Section 5.1)

We take a pretrained transformer (DeBERTa-v3-large) and fine-tune it on our data using LoRA, which only trains a small set of extra weights instead of the whole model - much faster and needs less GPU memory.

In [ ]:
# MCQDataset to convert Hugging face dataset into pytorch tensors and ready data to feed transformer model

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
TrainingArguments, Trainer, EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset

MODEL_NAME = "microsoft/deberta-v3-large"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

class MCQDataset(TorchDataset):
    # Plain PyTorch dataset
    def __init__(self, hf_dataset):
        d = hf_dataset.to_dict()
        self.input_ids = torch.tensor(d["input_ids"])
        self.attention_mask = torch.tensor(d["attention_mask"])
        self.token_type_ids = torch.tensor(d["token_type_ids"]) if "token_type_ids" in d else None
        self.labels = torch.tensor(d["labels"]) if "labels" in d else None

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        item = {"input_ids": self.input_ids[i], "attention_mask": self.attention_mask[i]}
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[i]
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item

hf_train = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True).rename_column("label", "labels")
hf_val   = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize, batched=True).rename_column("label", "labels")

torch_train = MCQDataset(hf_train)
torch_val   = MCQDataset(hf_val)


In [ ]:
class WeightedTrainer(Trainer):
    # Weights the 'correct answer' higher (1 correct : 4 wrong per question)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        weight = torch.tensor([1.0, 4.0]).to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, zero_division=0),
        "pred_positive_rate": preds.mean(),   # sanity check: should land near 0.2
    }


In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["query_proj", "key_proj", "value_proj", "dense"],
)
lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

EPOCHS = 6
steps_per_epoch = max(1, len(torch_train) // (8 * 4))
warmup_steps = int(0.1 * steps_per_epoch * EPOCHS)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/lora_outputs",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_steps=warmup_steps,
    fp16=False, bf16=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="model1-deberta-lora",
)

trainer = WeightedTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=torch_train,
    eval_dataset=torch_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


In [ ]:
def lora_score(dataset):
    # Run the fine-tuned model over a dataset
    lora_model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=32)
    probs = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            out = lora_model(**batch)
            probs.extend(F.softmax(out.logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.array(probs)

val_probs_lora = lora_score(torch_val)

metrics_lora = evaluate_model("Model 1 - DeBERTa+LoRA (fine-tuned transformer)", val_df, val_probs_lora, val_answers)
log_run("model1-deberta-lora", metrics_lora, config={"model": MODEL_NAME, "lora_r": 32, "epochs": EPOCHS})


## Shared Setup for Models 2 & 3 (built from scratch)

In [ ]:
SCRATCH_MAX_LEN = 64  # shorter than the transformer's 256 - these models are much lighter
VOCAB_SIZE = tokenizer.vocab_size

def encode_scratch(texts):
    enc = tokenizer(list(texts), truncation=True, padding="max_length", max_length=SCRATCH_MAX_LEN)
    return np.array(enc["input_ids"])

X_train, y_train = encode_scratch(train_df["text"]), train_df["label"].values
X_val,   y_val   = encode_scratch(val_df["text"]),   val_df["label"].values
X_test           = encode_scratch(test_processed["text"])

## Model 2: Custom Deep Learning Model - Text-CNN

A small CNN we build ourselves (no pretrained weights). It slides filters over the text to pick up short word patterns, then keeps only the strongest signal from each filter.

In [ ]:
# Simple architecture diagram (Figure 1) — one box per layer, top to bottom
layers = [
    "Input text (64 words)",
    "Embedding (vocab -> 128)",
    "Conv1D, kernels 3/4/5 (100 filters each)",
    "ReLU + Max-Pooling",
    "Concatenate (300 features)",
    "Dropout (0.3)",
    "Linear (300 -> 2)",
]

fig, ax = plt.subplots(figsize=(6, 7))
n = len(layers)
for i, layer in enumerate(layers):
    y = n - i
    ax.text(0.5, y, layer, ha="center", va="center", fontsize=10, color="white",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#4C72B0", edgecolor="black"))
    if i > 0:
        ax.annotate("", xy=(0.5, y + 0.35), xytext=(0.5, y + 0.65),
                     arrowprops=dict(arrowstyle="->"))

ax.set_xlim(0, 1)
ax.set_ylim(0, n + 1)
ax.axis("off")
ax.set_title("Figure 1: Text-CNN architecture")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figure1_textcnn_architecture.png", dpi=150)
plt.show()


In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=100, kernel_sizes=(3, 4, 5)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=tokenizer.pad_token_id)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 2)

    def forward(self, input_ids):
        x = self.embedding(input_ids).transpose(1, 2)      # (batch, embed_dim, seq)
        pooled = [F.max_pool1d(F.relu(conv(x)), x.size(2)).squeeze(2) for conv in self.convs]
        features = torch.cat(pooled, dim=1)                  # (batch, 300)
        return self.fc(self.dropout(features))


## Model 3: RNN-Based Model - LSTM

Also built from scratch, using the same vocabulary as Model 2. Instead of looking at local word patterns like the CNN, the LSTM reads the text word by word and remembers what it has seen so far.

In [ ]:
# Simple architecture diagram (Figure 2) — same style as the Text-CNN diagram above
layers_lstm = [
    "Input text (64 words)",
    "Embedding (vocab -> 128)",
    "LSTM (hidden size 128)",
    "Last hidden state",
    "Dropout (0.3)",
    "Linear (128 -> 2)",
]

fig, ax = plt.subplots(figsize=(6, 6))
n = len(layers_lstm)
for i, layer in enumerate(layers_lstm):
    y = n - i
    ax.text(0.5, y, layer, ha="center", va="center", fontsize=10, color="white",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#55A868", edgecolor="black"))
    if i > 0:
        ax.annotate("", xy=(0.5, y + 0.35), xytext=(0.5, y + 0.65),
                     arrowprops=dict(arrowstyle="->"))

ax.set_xlim(0, 1)
ax.set_ylim(0, n + 1)
ax.axis("off")
ax.set_title("Figure 2: LSTM architecture")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figure2_lstm_architecture.png", dpi=150)
plt.show()


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=tokenizer.pad_token_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        _, (hidden, _) = self.lstm(x)              # h_n: last hidden state
        return self.fc(self.dropout(hidden[-1]))


### Train & evaluate Models 2 and 3

Same simple training loop for both, since they only differ in architecture.

In [ ]:
def train_scratch_model(model, run_name, epochs=8, lr=1e-3):
    """Train a from-scratch model and log train/val loss to wandb every epoch"""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 4.0]).to(device))  # correct answer weighted higher

    train_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.long))
    val_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.long))
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=64)

    run = wandb.init(project=WANDB_PROJECT, name=run_name, config={"epochs": epochs, "lr": lr})

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                val_loss += loss_fn(model(x), y).item()
        val_loss /= len(val_loader)

        wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"[{run_name}] epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    return model, run


def score_model(model, X):
    model.eval()
    loader = torch.utils.data.DataLoader(torch.tensor(X, dtype=torch.long), batch_size=64)
    probs = []
    with torch.no_grad():
        for x in loader:
            logits = model(x.to(device))
            probs.extend(F.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.array(probs)


# Model 2 - Text-CNN
textcnn_model, run_cnn = train_scratch_model(TextCNN(vocab_size=VOCAB_SIZE), "model2-textcnn")
val_probs_cnn  = score_model(textcnn_model, X_val)
test_probs_cnn = score_model(textcnn_model, X_test)
metrics_cnn = evaluate_model("Model 2 - Text-CNN (from scratch)", val_df, val_probs_cnn, val_answers)
wandb.log(metrics_cnn)
run_cnn.finish()

# Model 3 - LSTM
lstm_model, run_lstm = train_scratch_model(LSTMClassifier(vocab_size=VOCAB_SIZE), "model3-lstm")
val_probs_lstm  = score_model(lstm_model, X_val)
test_probs_lstm = score_model(lstm_model, X_test)
metrics_lstm = evaluate_model("Model 3 - LSTM (from scratch)", val_df, val_probs_lstm, val_answers)
wandb.log(metrics_lstm)
run_lstm.finish()


## Milestone 5: Ensembling


In [ ]:
summary = pd.DataFrame([
    {"model": "Model 1 - DeBERTa-v3 + LoRA (fine-tuned transformer)", **metrics_lora},
    {"model": "Model 2 - Text-CNN (custom, from scratch)",            **metrics_cnn},
    {"model": "Model 3 - LSTM (RNN, from scratch)",                   **metrics_lstm},
]).sort_values("val_map3", ascending=False).reset_index(drop=True)

summary.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)

run = wandb.init(project=WANDB_PROJECT, name="model-comparison")
wandb.log({"comparison_table": wandb.Table(dataframe=summary)})
run.finish()

summary


## Final Submission


In [ ]:
hf_test = Dataset.from_pandas(test_processed[["text"]]).map(tokenize, batched=True)
torch_test = MCQDataset(hf_test)
test_probs_lora = lora_score(torch_test)

# Simple weighted average — LoRA is the strongest model, so it counts for more
WEIGHTS = {"lora": 0.65, "cnn": 0.2, "lstm": 0.15}

val_ensemble = (
    WEIGHTS["lora"] * normalize(val_probs_lora) +
    WEIGHTS["cnn"]  * normalize(val_probs_cnn) +
    WEIGHTS["lstm"] * normalize(val_probs_lstm)
)
metrics_ensemble = evaluate_model("Ensemble (3 models)", val_df, val_ensemble, val_answers)
log_run("final-ensemble", metrics_ensemble, config=WEIGHTS)

test_ensemble = (
    WEIGHTS["lora"] * normalize(test_probs_lora) +
    WEIGHTS["cnn"]  * normalize(test_probs_cnn) +
    WEIGHTS["lstm"] * normalize(test_probs_lstm)
)

final_submission = scores_to_submission(
    test_processed["id"].values,
    test_processed["option"].values,
    test_ensemble,
    out_path=f"{OUT_DIR}/submission.csv",
)
print("Saved submission.csv")
final_submission.head(10)
